In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/mealrecplus/course_with_taxonomy.pkl
/kaggle/input/mealrecplus/MealRecPlus-main/MealRecPlus-main/README.md
/kaggle/input/mealrecplus/MealRecPlus-main/MealRecPlus-main/.gitattributes
/kaggle/input/mealrecplus/MealRecPlus-main/MealRecPlus-main/healthiness_eval.py
/kaggle/input/mealrecplus/MealRecPlus-main/MealRecPlus-main/MealRec+/data_load.py
/kaggle/input/mealrecplus/MealRecPlus-main/MealRecPlus-main/MealRec+/MealRec+L/meal_course.txt
/kaggle/input/mealrecplus/MealRecPlus-main/MealRecPlus-main/MealRec+/MealRec+L/user_meal_tune.txt
/kaggle/input/mealrecplus/MealRecPlus-main/MealRecPlus-main/MealRec+/MealRec+L/user_meal_test.txt
/kaggle/input/mealrecplus/MealRecPlus-main/MealRecPlus-main/MealRec+/MealRec+L/user_meal_train.txt
/kaggle/input/mealrecplus/MealRecPlus-main/MealRecPlus-main/MealRec+/MealRec+L/user_course.txt
/kaggle/input/mealrecplus/MealRecPlus-main/MealRecPlus-main/MealRec+/MealRec+L/course_category.txt
/kaggle/input/mealrecplus/MealRecPlus-main/MealRecPlus-mai

In [2]:
import pandas as pd
import numpy as np
from collections import defaultdict, Counter

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader


In [3]:
ROOT = "/kaggle/input/mealrecplus/MealRecPlus-main/MealRecPlus-main/MealRec+/MealRec+H"

PATH_MEAL_COURSE = f"{ROOT}/meal_course.txt"
PATH_USER_MEAL_TRAIN = f"{ROOT}/user_meal_train.txt"
PATH_USER_MEAL_TEST  = f"{ROOT}/user_meal_test.txt"
PATH_COURSE2INDEX = f"{ROOT}/meta_data/course2index.txt"
PATH_COURSE_CATEGORY = f"{ROOT}/course_category.txt"

PATH_TAXO = "/kaggle/input/mealrecplus/course_with_taxonomy.pkl"


In [4]:
meal_course = pd.read_csv(PATH_MEAL_COURSE, sep="\t", header=None, names=["meal_id", "course_index"])
meal_course["meal_id"] = meal_course["meal_id"].astype(int)
meal_course["course_index"] = meal_course["course_index"].astype(int)

meal_ids = meal_course["meal_id"].unique()
meal2idx = {m:i for i,m in enumerate(meal_ids)}
idx2meal = {i:m for m,i in meal2idx.items()}
n_meals = len(meal2idx)

meal_course["meal_idx"] = meal_course["meal_id"].map(meal2idx).astype(int)

meal_to_courses = [[] for _ in range(n_meals)]
for mi, ci in meal_course[["meal_idx","course_index"]].itertuples(index=False):
    meal_to_courses[int(mi)].append(int(ci))

max_course_index = max(ci for courses in meal_to_courses for ci in courses)
n_courses_index_space = max_course_index + 1

empty_meals = sum(1 for courses in meal_to_courses if len(courses)==0)

print("n_meals:", n_meals)
print("max_course_index:", max_course_index)
print("n_courses_index_space:", n_courses_index_space)
print("empty_meals:", empty_meals)


n_meals: 3817
max_course_index: 4804
n_courses_index_space: 4805
empty_meals: 0


In [5]:
train_um = pd.read_csv(PATH_USER_MEAL_TRAIN, sep="\t", header=None, names=["user_id","meal_id"])
train_um["user_id"] = train_um["user_id"].astype(int)
train_um["meal_id"] = train_um["meal_id"].astype(int)

user_ids = train_um["user_id"].unique()
user2idx = {u:i for i,u in enumerate(user_ids)}
idx2user = {i:u for u,i in user2idx.items()}
n_users = len(user2idx)

train_um["user_idx"] = train_um["user_id"].map(user2idx).astype(int)
train_um["meal_idx"] = train_um["meal_id"].map(meal2idx)
train_um = train_um.dropna(subset=["meal_idx"]).copy()
train_um["meal_idx"] = train_um["meal_idx"].astype(int)

print("n_users:", n_users, "train interactions:", len(train_um))


n_users: 1575 train interactions: 37413


In [6]:
course2index = pd.read_csv(PATH_COURSE2INDEX, sep="\t", header=None)
c0 = course2index.iloc[:,0].astype(int)
c1 = course2index.iloc[:,1].astype(int)

# tự nhận dạng cột nào là course_index
if c0.min() == 0:
    course_index = c0
    course_id = c1
else:
    course_id = c0
    course_index = c1

id2cindex = dict(zip(course_id, course_index))

course_df = pd.read_pickle(PATH_TAXO)
course_df["course_id"] = course_df["course_id"].astype(int)

def ensure_list(x):
    return x if isinstance(x, list) else []

for col in ["cuisine_ids","ingredient_ids","flavor_ids"]:
    course_df[col] = course_df[col].apply(ensure_list)

course_df["course_index"] = course_df["course_id"].map(id2cindex)
course_df = course_df.dropna(subset=["course_index"]).copy()
course_df["course_index"] = course_df["course_index"].astype(int)

taxo_by_cindex = dict(
    zip(course_df["course_index"].values,
        zip(course_df["cuisine_ids"].values,
            course_df["ingredient_ids"].values,
            course_df["flavor_ids"].values))
)

# vocab sizes = max_id+1
max_cui = max([max(v[0]) for v in taxo_by_cindex.values() if len(v[0])>0] + [-1]) + 1
max_ing = max([max(v[1]) for v in taxo_by_cindex.values() if len(v[1])>0] + [-1]) + 1
max_fla = max([max(v[2]) for v in taxo_by_cindex.values() if len(v[2])>0] + [-1]) + 1

# taxonomy coverage trên course_index xuất hiện trong meal_course
all_course_in_meals = {ci for courses in meal_to_courses for ci in courses}
coverage = sum(1 for ci in all_course_in_meals if ci in taxo_by_cindex)

print("taxo dict size:", len(taxo_by_cindex))
print("vocab (cui/ing/fla):", max_cui, max_ing, max_fla)
print("taxonomy coverage on meal courses:", coverage, "/", len(all_course_in_meals))


taxo dict size: 7280
vocab (cui/ing/fla): 8 12 7
taxonomy coverage on meal courses: 942 / 942


In [7]:
# course_index range check
assert max_course_index < n_courses_index_space

# taxonomy id range checks
max_cui_real = max([max(v[0]) for v in taxo_by_cindex.values() if len(v[0])>0] + [-1])
max_ing_real = max([max(v[1]) for v in taxo_by_cindex.values() if len(v[1])>0] + [-1])
max_fla_real = max([max(v[2]) for v in taxo_by_cindex.values() if len(v[2])>0] + [-1])

assert max_cui_real < max_cui
assert max_ing_real < max_ing
assert max_fla_real < max_fla

print("Index sanity checks passed ✅")


Index sanity checks passed ✅


In [8]:
class MealBPRDataset(Dataset):
    def __init__(self, df, n_meals):
        self.u = df["user_idx"].values
        self.m = df["meal_idx"].values
        self.n_meals = n_meals

        self.user_pos = {}
        for uu, mm in zip(self.u, self.m):
            self.user_pos.setdefault(int(uu), set()).add(int(mm))

    def __len__(self):
        return len(self.u)

    def __getitem__(self, idx):
        u = int(self.u[idx])
        pos = int(self.m[idx])
        # negative sampling
        while True:
            neg = np.random.randint(0, self.n_meals)
            if neg not in self.user_pos[u]:
                break
        return u, pos, neg

train_ds = MealBPRDataset(train_um, n_meals)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, drop_last=True)


In [9]:
class ComboTaxoBPR(nn.Module):
    def __init__(self, n_users, n_courses_index_space, d,
                 max_cui, max_ing, max_fla,
                 meal_to_courses, taxo_by_cindex):
        super().__init__()
        self.d = d
        self.meal_to_courses = meal_to_courses
        self.taxo_by_cindex = taxo_by_cindex

        self.user_emb = nn.Embedding(n_users, d)
        self.course_emb = nn.Embedding(n_courses_index_space, d)

        self.cui_emb = nn.Embedding(max(1, max_cui), d)
        self.ing_emb = nn.Embedding(max(1, max_ing), d)
        self.fla_emb = nn.Embedding(max(1, max_fla), d)

        for emb in [self.user_emb, self.course_emb, self.cui_emb, self.ing_emb, self.fla_emb]:
            nn.init.normal_(emb.weight, std=0.01)

    def _mean_emb(self, ids, emb, device):
        # lọc id an toàn
        ids = [x for x in ids if 0 <= x < emb.num_embeddings]
        if len(ids) == 0:
            return torch.zeros(self.d, device=device)
        t = torch.tensor(ids, device=device, dtype=torch.long)
        return emb(t).mean(dim=0)

    def encode_course(self, cindex, device):
        # cindex an toàn
        if not (0 <= cindex < self.course_emb.num_embeddings):
            return torch.zeros(self.d, device=device)

        base = self.course_emb(torch.tensor(cindex, device=device))
        cuis, ings, flas = self.taxo_by_cindex.get(cindex, ([], [], []))

        return (base
                + self._mean_emb(cuis, self.cui_emb, device)
                + self._mean_emb(ings, self.ing_emb, device)
                + self._mean_emb(flas, self.fla_emb, device))

    def encode_meal(self, meal_idx, device):
        courses = self.meal_to_courses[meal_idx]
        if len(courses) == 0:
            return torch.zeros(self.d, device=device)
        vecs = torch.stack([self.encode_course(ci, device) for ci in courses], dim=0)
        return vecs.mean(dim=0)

    def score(self, user_idx, meal_idx, device):
        u = self.user_emb(user_idx)  # [d]
        m = self.encode_meal(int(meal_idx.item()), device)  # [d]
        return (u * m).sum()

def bpr_loss(pos, neg):
    return -torch.log(torch.sigmoid(pos - neg) + 1e-8)


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ComboTaxoBPR(
    n_users=n_users,
    n_courses_index_space=n_courses_index_space,
    d=64,
    max_cui=max_cui, max_ing=max_ing, max_fla=max_fla,
    meal_to_courses=meal_to_courses,
    taxo_by_cindex=taxo_by_cindex
).to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(1, 31):
    model.train()
    total = 0.0

    for u, mp, mn in train_loader:
        u = u.to(device)
        mp = mp.to(device)
        mn = mn.to(device)

        loss = 0.0
        # chậm nhưng rõ ràng
        for i in range(u.size(0)):
            pos = model.score(u[i], mp[i], device)
            neg = model.score(u[i], mn[i], device)
            loss = loss + bpr_loss(pos, neg)
        loss = loss / u.size(0)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total += loss.item() * u.size(0)

    print(f"Epoch {epoch} | loss={total/len(train_ds):.4f}")


Epoch 1 | loss=0.6857
Epoch 2 | loss=0.6305
Epoch 3 | loss=0.5667
Epoch 4 | loss=0.5187
Epoch 5 | loss=0.4780
Epoch 6 | loss=0.4432
Epoch 7 | loss=0.4157
Epoch 8 | loss=0.3911
Epoch 9 | loss=0.3654
Epoch 10 | loss=0.3481
Epoch 11 | loss=0.3320
Epoch 12 | loss=0.3132
Epoch 13 | loss=0.2985
Epoch 14 | loss=0.2823
Epoch 15 | loss=0.2697
Epoch 16 | loss=0.2595
Epoch 17 | loss=0.2489
Epoch 18 | loss=0.2427
Epoch 19 | loss=0.2302
Epoch 20 | loss=0.2193
Epoch 21 | loss=0.2141
Epoch 22 | loss=0.2019
Epoch 23 | loss=0.1975
Epoch 24 | loss=0.1890
Epoch 25 | loss=0.1859
Epoch 26 | loss=0.1794
Epoch 27 | loss=0.1732
Epoch 28 | loss=0.1708
Epoch 29 | loss=0.1625
Epoch 30 | loss=0.1594


In [11]:
test_um = pd.read_csv(PATH_USER_MEAL_TEST, sep="\t", header=None, names=["user_id","meal_id"])
test_um["user_id"] = test_um["user_id"].astype(int)
test_um["meal_id"] = test_um["meal_id"].astype(int)

test_um["user_idx"] = test_um["user_id"].map(user2idx)
test_um["meal_idx"] = test_um["meal_id"].map(meal2idx)
test_um = test_um.dropna(subset=["user_idx","meal_idx"]).copy()
test_um["user_idx"] = test_um["user_idx"].astype(int)
test_um["meal_idx"] = test_um["meal_idx"].astype(int)

gt = test_um.groupby("user_idx")["meal_idx"].apply(set).to_dict()
print("n_eval_users:", len(gt))


n_eval_users: 1114


In [12]:
import math

def precision_at_k(preds, truth, k):
    preds = preds[:k]
    hit = sum(1 for p in preds if p in truth)
    return hit / k

def recall_at_k(preds, truth, k):
    preds = preds[:k]
    if len(truth) == 0:
        return 0.0
    hit = sum(1 for p in preds if p in truth)
    return hit / len(truth)

def ndcg_at_k(preds, truth, k):
    preds = preds[:k]
    dcg = 0.0
    for i, p in enumerate(preds):
        if p in truth:
            dcg += 1.0 / math.log2(i + 2)
    ideal = min(len(truth), k)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(ideal))
    return 0.0 if idcg == 0 else dcg / idcg


In [13]:
@torch.no_grad()
def precompute_meal_embeddings(model, n_meals, device):
    model.eval()
    meal_embs = torch.zeros((n_meals, model.d), device=device)
    for m in range(n_meals):
        meal_embs[m] = model.encode_meal(m, device)
    return meal_embs

@torch.no_grad()
def recommend_topk_fast(user_idx, k, user_embs, meal_embs, seen_train, device):
    scores = user_embs[user_idx].to(device) @ meal_embs.T  # [n_meals]
    seen = seen_train.get(user_idx, set())
    if len(seen) > 0:
        scores[list(seen)] = -1e9
    return torch.topk(scores, k=k).indices.cpu().numpy().tolist()

def evaluate_fast(model, gt, k, device):
    model.eval()
    meal_embs = precompute_meal_embeddings(model, n_meals, device)
    user_embs = model.user_emb.weight.detach()

    ps, rs, ns = [], [], []
    for u in gt.keys():
        truth = gt[u]
        preds = recommend_topk_fast(u, k, user_embs, meal_embs, train_ds.user_pos, device)
        ps.append(precision_at_k(preds, truth, k))
        rs.append(recall_at_k(preds, truth, k))
        ns.append(ndcg_at_k(preds, truth, k))

    return {
        f"Precision@{k}": float(np.mean(ps)),
        f"Recall@{k}": float(np.mean(rs)),
        f"NDCG@{k}": float(np.mean(ns)),
        "n_eval_users": len(gt)
    }

metrics = evaluate_fast(model, gt, k=10, device=device)
print(metrics)


{'Precision@10': 0.04488330341113106, 'Recall@10': 0.14531272334385328, 'NDCG@10': 0.1133652699157216, 'n_eval_users': 1114}


In [14]:
SAVE_PATH = "/kaggle/working/combotaxobpr.pth"
torch.save(model.state_dict(), SAVE_PATH)

print("Model saved to:", SAVE_PATH)


Model saved to: /kaggle/working/combotaxobpr.pth
